# Corpus Au@SiO2 YOLO-Seg Colab Runner

Upload one `corpus_colab_training_bundle.zip` or point to a prepared bundle/dataset in Google Drive, then run all cells. Default behavior runs a 1-epoch smoke test.

In [ ]:
!pip install -q ultralytics
import pathlib, zipfile, shutil, subprocess, sys, os
from google.colab import files, drive

# Options: 'upload', 'drive_zip', or 'drive_dataset'
INPUT_MODE = 'upload'
DRIVE_ZIP = '/content/drive/MyDrive/corpus_colab_training_bundle.zip'
DRIVE_DATASET = '/content/drive/MyDrive/corpus_yolo_seg'
RUN_FULL_TRAINING = False

SMOKE_ARGS = ['--smoke-only']
FULL_ARGS = ['--full', '--epochs', '75', '--imgsz', '1024', '--batch', '4']

In [ ]:
bundle_path = ''
dataset_path = ''

if INPUT_MODE == 'upload':
    uploaded = files.upload()
    zips = [name for name in uploaded if name.endswith('.zip')]
    assert zips, 'Upload corpus_colab_training_bundle.zip'
    bundle_path = '/content/' + zips[0]
elif INPUT_MODE == 'drive_zip':
    drive.mount('/content/drive')
    bundle_path = DRIVE_ZIP
elif INPUT_MODE == 'drive_dataset':
    drive.mount('/content/drive')
    dataset_path = DRIVE_DATASET
else:
    raise ValueError(INPUT_MODE)

print('bundle:', bundle_path)
print('dataset:', dataset_path)

In [ ]:
runner_path = pathlib.Path('/content/colab_run_training.py')
if bundle_path:
    with zipfile.ZipFile(bundle_path, 'r') as zf:
        zf.extract('training/colab_run_training.py', '/content/_runner_extract')
    shutil.copy('/content/_runner_extract/training/colab_run_training.py', runner_path)
else:
    # If using a Drive dataset folder, upload/copy this script next to the notebook if needed.
    possible = pathlib.Path('/content/drive/MyDrive/colab_run_training.py')
    assert possible.exists(), 'Provide colab_run_training.py in Drive or use a ZIP bundle.'
    shutil.copy(possible, runner_path)

cmd = [sys.executable, str(runner_path)]
if bundle_path:
    cmd += ['--bundle', bundle_path]
else:
    cmd += ['--dataset', dataset_path]
cmd += FULL_ARGS if RUN_FULL_TRAINING else SMOKE_ARGS
print(' '.join(cmd))
try:
    subprocess.check_call(cmd)
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        f"Training script failed (exit {e.returncode}). "
        "Scroll up in this cell to see the full error from colab_run_training.py."
    ) from None

## Full Training

After the smoke run passes, set `RUN_FULL_TRAINING = True` in the first cell and run all cells again. Outputs are saved to `/content/drive/MyDrive/corpus_runs` when Drive is mounted, otherwise `/content/corpus_runs`.

In [ ]:
import pathlib
runs = (pathlib.Path('/content/drive/MyDrive/corpus_runs')
        if pathlib.Path('/content/drive/MyDrive').exists()
        else pathlib.Path('/content/corpus_runs'))
weights = sorted(runs.rglob('best.pt'))
if weights:
    print("Best weights saved to:")
    for w in weights:
        print(" ", w)
else:
    print("No weights found in", runs)